In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from glam import Geocoder, download_dependencies
from pathlib import Path

import requests
import geopandas as gpd

In [39]:
import csv
import io

with open("DVRS270826.txt", encoding="utf-16") as f:
    lines = f.read().splitlines()

header, *rows = lines
expected_cols = len(header.split("|"))


def unwrap(line):
    if len(line) >= 2 and line[0] == '"' and line[-1] == '"':
        return line[1:-1]
    return line


cleaned = [unwrap(line) for line in rows]

# Drop genuinely malformed rows (e.g. a stray "#NAME?" Excel-error row) instead
# of letting them silently misalign columns.
good_rows = [line for line in cleaned if len(line.split("|")) == expected_cols]
n_dropped = len(cleaned) - len(good_rows)
if n_dropped:
    print(f"Dropping {n_dropped} malformed row(s) that don't split into {expected_cols} fields")

csv_text = "\n".join([header] + good_rows)
df = pd.read_csv(io.StringIO(csv_text), sep="|", dtype=str, quoting=csv.QUOTE_NONE)


Dropping 1 malformed row(s) that don't split into 41 fields


In [40]:
# Normalize column names

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

In [41]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 92010 entries, 0 to 92009
Data columns (total 41 columns):
 #   Column                            Non-Null Count  Dtype
---  ------                            --------------  -----
 0   valuation_roll_number             92010 non-null  str  
 1   valuation_number_assessment       92010 non-null  str  
 2   valuation_number_suffix           14153 non-null  str  
 3   sale_date                         92010 non-null  str  
 4   district_code                     92010 non-null  str  
 5   sale_type                         92009 non-null  str  
 6   sales_group                       92010 non-null  str  
 7   sale_tenure                       92010 non-null  str  
 8   price_value_relationship          92010 non-null  str  
 9   sale_price_gross                  92010 non-null  str  
 10  sale_price_net                    92010 non-null  str  
 11  sale_price_chattels               92010 non-null  str  
 12  sale_price_other                  92010 non

In [42]:
# Sanity check
print(df.shape)
print(df[["sale_date", "sale_price_gross", "sale_price_net"]].isna().sum())

(92010, 41)
sale_date           0
sale_price_gross    0
sale_price_net      0
dtype: int64


## Geocode via LINZ Address Matching

In [43]:
def download_glam_dependencies(deps_directory: str) -> Path:
    deps_path = Path(deps_directory)
    marker = deps_path / "nz-street-address.csv"

    if marker.exists():
        print(f"glam dependencies already present in {deps_path}, skipping download")
    else:
        download_dependencies(str(deps_path))


In [44]:
deps_dir = Path("./glam-deps")
if not (deps_dir / "nz-street-address.csv").exists():
    download_glam_dependencies(str(deps_dir))

gc = Geocoder("./glam-deps", matcher="tfidf", parser="rnn")


17:29 - GLAM - INFO - Loading NZSA data from glam-deps\nz-street-address.csv
17:29 - GLAM - INFO - Using existing postcodes


In [45]:
# Subsetting df for experimental
df = df[:10000]

In [46]:
# Build a search address per row from what we have (no suburb available yet).
addresses = (
    df["situation_number"].fillna("") + " " + df["situation_name"].fillna("")
).str.strip()

results = gc.geocode_addresses(addresses.tolist())

df["geocode_confidence"] = [r.confidence for r in results]
df["longitude"] = [r.matched_address.shape_X if r.matched_address else None for r in results]
df["latitude"] = [r.matched_address.shape_Y if r.matched_address else None for r in results]
df["matched_suburb"] = [r.matched_address.suburb_locality if r.matched_address else None for r in results]
df["matched_postcode"] = [r.matched_address.postcode if r.matched_address else None for r in results]

print(df["geocode_confidence"].describe())
print("no match at all:", df["longitude"].isna().sum())


17:29 - GLAM - INFO - Loading dependencies for TFIDF parser
d:\Codes\COMPSCI_760_Group_3_Intelligent_Real_Estate_Price_Prediction_Model\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Codes\COMPSCI_760_Group_3_Intelligent_Real_Estate_Price_Prediction_Model\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
17:29 - GLAM - INFO - Beginning

count    10000.000000
mean         0.678806
std          0.078183
min          0.432854
25%          0.623846
50%          0.677456
75%          0.727202
max          0.914609
Name: geocode_confidence, dtype: float64
no match at all: 0


## Flag properties inside Auckland Council flood plain hazard zones

In [47]:

# Auckland Council open data hazard layers, served as public ArcGIS
# FeatureServers (no API key needed). Each is paginated and cached locally as
# GeoJSON under hazard_data/ (created automatically if missing) so we only
# hit the network once per layer.
def fetch_arcgis_layer(
    query_url: str,
    cache_path: str,
    out_fields: str = "*",
    page_size: int = 2000,
) -> gpd.GeoDataFrame:
    cache_file = Path(cache_path)
    if cache_file.exists():
        return gpd.read_file(cache_file)

    cache_file.parent.mkdir(parents=True, exist_ok=True)

    offset = 0
    features = []
    while True:
        resp = requests.get(
            query_url,
            params={
                "where": "1=1",
                "outFields": out_fields,
                "outSR": 4326,  # reproject server-side to match our WGS84 lon/lat
                "resultOffset": offset,
                "resultRecordCount": page_size,
                "f": "geojson",
            },
            timeout=60,
        )
        resp.raise_for_status()
        page = resp.json()["features"]
        if not page:
            break
        features.extend(page)
        offset += page_size

    gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
    gdf.to_file(cache_file, driver="GeoJSON")
    return gdf


flood_plains = fetch_arcgis_layer(
    "https://services1.arcgis.com/n4yPwebTjJCmXB6W/arcgis/rest/services/Flood_Plains/FeatureServer/0/query",
    "./hazard_data/flood_plains.geojson",
    out_fields="Hazard,YEAR_PRODUCED",
)
print(f"Loaded {len(flood_plains)} flood plain polygons")


Loaded 12629 flood plain polygons


In [48]:
# Only rows that actually got coordinates can be checked; leave the rest as
# <NA> (unknown) rather than silently defaulting them to "not flooded".
geocoded = df.dropna(subset=["longitude", "latitude"])
points_gdf = gpd.GeoDataFrame(
    geocoded[[]],
    geometry=gpd.points_from_xy(geocoded["longitude"], geocoded["latitude"]),
    crs="EPSG:4326",
)


def flag_within_hazard_zone(
    target: pd.DataFrame,
    points: gpd.GeoDataFrame,
    hazard_gdf: gpd.GeoDataFrame,
    column: str,
) -> None:
    """Set target[column] True/False/<NA> for whether each geocoded row falls
    inside any polygon of hazard_gdf. Rows with no coordinates (not in
    `points`) stay <NA> instead of being counted as "not in the hazard zone".
    """
    matches = gpd.sjoin(points, hazard_gdf, how="inner", predicate="within")
    matched_ids = set(matches.index)

    target[column] = pd.NA
    target.loc[points.index, column] = False
    target.loc[target.index.isin(matched_ids), column] = True


flag_within_hazard_zone(df, points_gdf, flood_plains, "in_flood_plain")
print(df["in_flood_plain"].value_counts(dropna=False))


in_flood_plain
False    9504
True      496
Name: count, dtype: int64


In [49]:
# Flood Sensitive Areas: broader ponding/overland-flow hazard layer than the
# mapped Flood Plains extents. Uses the cached hazard_data/flood_sensitive_area.geojson
# if already present (e.g. your manual export); otherwise downloads and caches it
# the same way as flood_plains above.
flood_sensitive = fetch_arcgis_layer(
    "https://services1.arcgis.com/n4yPwebTjJCmXB6W/arcgis/rest/services/Flood_Sensitive_Areas/FeatureServer/0/query",
    "./hazard_data/flood_sensitive_area.geojson",
    out_fields="Hazard,YEAR_PRODUCED",
)
print(f"Loaded {len(flood_sensitive)} flood sensitive area polygons")

flag_within_hazard_zone(df, points_gdf, flood_sensitive, "in_flood_sensitive_area")
print(df["in_flood_sensitive_area"].value_counts(dropna=False))


Loaded 859 flood sensitive area polygons
in_flood_sensitive_area
False    9936
True       64
Name: count, dtype: int64
